# ⚙️ Fase 2 — Preprocessing & Feature Engineering
### Credit Card Fraud Detection · Omar Mora Flores

**Meta:** limpiar, enriquecer y preparar el dataset para modelado, dejando 4 arrays listos en
`data/splits.pkl`.

Decisiones clave (justificadas por el EDA de la Fase 1):
- **Eliminar 1,081 duplicados** detectados en el EDA (antes del split, para no filtrar info).
- **Feature engineering:** `Hour`, `Amount_log`, `Is_night`.
- **Escalado robusto** de `Amount_log` y `Hour` — *ajustado solo en train* (sin data leakage).
- **Split estratificado 80/20** para preservar el 0.173% de fraude.
- **SMOTE solo en train** — nunca en test.

In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

ROOT = Path.cwd()
while not (ROOT / "data" / "creditcard.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data" / "creditcard.csv"

RANDOM_STATE = 42
df = pd.read_csv(DATA)
print("Shape original:", df.shape)

Shape original: (284807, 31)


## 2.1 Limpieza — eliminación de duplicados

El EDA reveló **1,081 filas duplicadas** (0 nulos). Las eliminamos **antes** del split: si
quedaran, copias idénticas podrían caer a la vez en train y test, contaminando la evaluación.

In [2]:
n_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados eliminados: {n_antes - len(df):,}")
print(f"Shape tras limpieza  : {df.shape}")
print(f"Nulos restantes      : {int(df.isnull().sum().sum())}")

Duplicados eliminados: 1,081
Shape tras limpieza  : (283726, 31)
Nulos restantes      : 0


## 2.2 Feature Engineering

- **`Hour`** — hora del día (0–24) desde `Time`.
- **`Amount_log`** — `log1p(Amount)` para reducir el sesgo a la derecha visto en el EDA.
- **`Is_night`** — bandera 1 si la transacción ocurre entre 22:00–06:00 (el EDA mostró que la
  *tasa* de fraude sube de madrugada).

Luego se eliminan `Time` y `Amount`, reemplazadas por sus versiones derivadas.

In [3]:
df["Hour"] = (df["Time"] / 3600) % 24
df["Amount_log"] = np.log1p(df["Amount"])
df["Is_night"] = df["Hour"].apply(lambda h: 1 if (h >= 22 or h <= 6) else 0)

df = df.drop(columns=["Time", "Amount"])

print("Nuevas columnas creadas: Hour, Amount_log, Is_night")
print("% de transacciones nocturnas:", round(100 * df["Is_night"].mean(), 1), "%")
df[["Hour", "Amount_log", "Is_night", "Class"]].head()

Nuevas columnas creadas: Hour, Amount_log, Is_night
% de transacciones nocturnas: 17.7 %


,Hour,Amount_log,Is_night,Class
0,0.000000,5.014760,1,0
1,0.000000,1.305626,1,0
2,0.000278,5.939276,1,0
3,0.000278,4.824306,1,0
4,0.000556,4.262539,1,0


## 2.3 Split estratificado train/test

> **Mejora sobre el orden del roadmap:** hacemos el **split antes del escalado**. Ajustar el
> `RobustScaler` con datos de test sería *data leakage*. Por eso: split → escalar (fit en
> train) → SMOTE.

`stratify=y` preserva la proporción 0.173% de fraude en ambos conjuntos.

In [4]:
X = df.drop(columns=["Class"])
y = df["Class"]
FEATURES = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape} | fraude: {y_train.sum()} ({100*y_train.mean():.4f}%)")
print(f"Test : {X_test.shape} | fraude: {y_test.sum()} ({100*y_test.mean():.4f}%)")

Train: (226980, 31) | fraude: 378 (0.1665%)
Test : (56746, 31) | fraude: 95 (0.1674%)


## 2.4 Escalado robusto (fit solo en train)

`Amount_log` y `Hour` están en escalas distintas a las features `V1`–`V28` (que ya vienen
normalizadas por PCA). Aplicamos `RobustScaler` (robusto a outliers) **solo a esas dos
columnas**, ajustándolo únicamente con el train. `Is_night` es binaria y no se escala.

In [5]:
cols_scale = ["Amount_log", "Hour"]
scaler = RobustScaler()

X_train = X_train.copy()
X_test = X_test.copy()
X_train[cols_scale] = scaler.fit_transform(X_train[cols_scale])
X_test[cols_scale] = scaler.transform(X_test[cols_scale])

print("RobustScaler ajustado en train y aplicado a:", cols_scale)
print("V1–V28 NO se reescalan (ya vienen escaladas por PCA).")
X_train[cols_scale].describe().round(3)

RobustScaler ajustado en train y aplicado a: ['Amount_log', 'Hour']
V1–V28 NO se reescalan (ya vienen escaladas por PCA).


,Amount_log,Hour
count,226980.000,226980.000
mean,0.007,-0.053
std,0.671,0.669
min,-1.272,-1.720
25%,-0.502,-0.505
50%,0.000,0.000
75%,0.498,0.495
max,2.735,1.029


## 2.5 Balanceo de clases — SMOTE (solo en train)

SMOTE genera ejemplos sintéticos de la clase minoritaria interpolando vecinos. Se aplica
**solo al train**: usarlo en el test inventaría fraudes que nunca ocurrieron y daría métricas
infladas (data leakage). El test queda con su distribución real.

In [6]:
print("Antes de SMOTE :", dict(y_train.value_counts().sort_index()))

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Después de SMOTE:", dict(pd.Series(y_train_res).value_counts().sort_index()))
print("Test (intacto)  :", dict(y_test.value_counts().sort_index()))

Antes de SMOTE : {0: np.int64(226602), 1: np.int64(378)}


Después de SMOTE: {0: np.int64(226602), 1: np.int64(226602)}
Test (intacto)  : {0: np.int64(56651), 1: np.int64(95)}


## 2.6 Guardar splits procesados

Serializamos en `data/splits.pkl` los 4 arrays + la lista de features y el `scaler`
(necesarios para el dashboard de la Fase 4).

In [7]:
splits = {
    "X_train": X_train_res,
    "X_test": X_test,
    "y_train": y_train_res,
    "y_test": y_test,
    "features": FEATURES,
    "scaler": scaler,
    "cols_scaled": cols_scale,
}

out = ROOT / "data" / "splits.pkl"
with open(out, "wb") as f:
    pickle.dump(splits, f)

print("Guardado:", out)
print(f"X_train: {X_train_res.shape} | X_test: {X_test.shape}")
print("Features:", len(FEATURES))
print("\n➡️ Siguiente fase: 03_modeling.ipynb")

Guardado: D:\Proyectos\Data\fraud-detection\data\splits.pkl
X_train: (453204, 31) | X_test: (56746, 31)
Features: 31

➡️ Siguiente fase: 03_modeling.ipynb
